In [1]:
from operator import truth

from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from array import array
import os
from PIL import Image
import sys
import time

In [1]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials
from array import array
import os
from PIL import Image
import sys
import time

'''
Authenticate
Authenticates your credentials and creates a client.
'''
from config import VISION_KEY, VISION_ENDPOINT
subscription_key = VISION_KEY
endpoint = VISION_ENDPOINT
computervision_client = ComputerVisionClient(endpoint, CognitiveServicesCredentials(subscription_key))
'''
END - Authenticate
'''

'\nEND - Authenticate\n'

In [47]:
# img = open("data/test1.png", "rb")
img = open("data/test2.jpeg", "rb")
read_response = computervision_client.read_in_stream(
    image=img,
    mode="Printed",
    raw=True
)
# print(read_response.as_dict())

operation_id = read_response.headers['Operation-Location'].split('/')[-1]
while True:
    read_result = computervision_client.get_read_result(operation_id)
    if read_result.status not in ['notStarted', 'running']:
        break
    time.sleep(1)

# Print the detected text, line by line
result = []
if read_result.status == OperationStatusCodes.succeeded:
    for text_result in read_result.analyze_result.read_results:
        for line in text_result.lines:
            print(line.text)
            result.append(line.text)

print()

Lucces in resolvarea
TEMELOR la
LABORA toarele de
Inteligenta Artificialà!



In [48]:
# get/define the ground truth
# groundTruth = ["Google Cloud", "Platform"]
groundTruth = ["Succes in rezolvarea", "tEMELOR la", "LABORAtoaree de", "Inteligenta Artificiala!"]

# compute the performance
noOfCorrectLines = sum(i == j for i, j in zip(result, groundTruth))
print(noOfCorrectLines)

0


In [50]:
import re

#cate caractere corecte in pozitia corecta 

def how_far_off(result, groundTruth):
    dist = 0
    
    result_list = []
    truth_list = []
    for line in result:
        words = re.findall(r'\w+', line)
        result_list.append(words)
        
    for line in groundTruth:
        words = re.findall(r'\w+', line)
        truth_list.append(words)
        
    for word_result, word_truth in zip(result_list, truth_list):
        for w1, w2 in zip(word_result, word_truth):
            if w1 != w2:
                dist += 1
                
    return dist

In [54]:
def read_text_from_image(img):
    read_response = computervision_client.read_in_stream(
        image=img,
        mode="Printed",
        raw=True
    )
    operation_id = read_response.headers['Operation-Location'].split('/')[-1]
    while True:
        read_result = computervision_client.get_read_result(operation_id)
        if read_result.status not in ['notStarted', 'running']:
            break
        time.sleep(1)

    result = []
    if read_result.status == OperationStatusCodes.succeeded:
        for text_result in read_result.analyze_result.read_results:
            for line in text_result.lines:
                result.append(line.text)
    
    return result

print(read_text_from_image(open("data/test2.jpeg", "rb")))

import re

result_list = []
    
for line in result:
    words = re.findall(r'\w+', line)
    result_list.append(words)
    
print(result_list)

groundTruth = ["Succes in rezolvarea", "tEMELOR la", "LABORAtoaree de", "Inteligenta Artificiala!"]

print(how_far_off(read_text_from_image(open("data/test2.jpeg", "rb")), groundTruth))

['Lucces in resolvarea', 'TEMELOR la', 'LABORA toarele de', 'Inteligenta Artificialà!']
[['Lucces', 'in', 'resolvarea'], ['TEMELOR', 'la'], ['LABORA', 'toarele', 'de'], ['Inteligenta', 'Artificialà']]
6


In [15]:

#cate caractere corecte in pozitia corecta 
#same as hamming 
def how_far_off2(img, groundTruth):
    result = read_text_from_image(open(img, "rb"))
    dist = 0
    
    result_list = []
    truth_list = []
    for line in result:
        words = re.findall(r'\w+', line)
        result_list.append(words)
        
    for line in groundTruth:
        words = re.findall(r'\w+', line)
        truth_list.append(words)
        
    for word_result, word_truth in zip(result_list, truth_list):
        for w1, w2 in zip(word_result, word_truth):
            if w1 != w2:
                dist += 1
                
    return dist

how_far_off2("data/test2.jpeg", groundTruth)
    

6

In [ ]:
#CE SE CERE LAB 3 (sper)

In [1]:
import os
import time
import requests
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes
from msrest.authentication import CognitiveServicesCredentials

# Distance metric libraries
from Levenshtein import distance as levenshtein_distance
from jellyfish import jaro_winkler_similarity
import difflib  # for LCS

In [2]:
def read_text_from_image(img):
    read_response = computervision_client.read_in_stream(
        image=img,
        mode="Printed",
        raw=True
    )
    operation_id = read_response.headers['Operation-Location'].split('/')[-1]
    while True:
        read_result = computervision_client.get_read_result(operation_id)
        if read_result.status not in ['notStarted', 'running']:
            break
        time.sleep(1)

    result = []
    if read_result.status == OperationStatusCodes.succeeded:
        for text_result in read_result.analyze_result.read_results:
            for line in text_result.lines:
                result.append(line.text)
    
    return result

In [20]:
def hamming_distance(s1: str, s2: str) -> int:
    """Hamming distance — strings must be equal length."""
    if len(s1) != len(s2):
        raise ValueError(f"Hamming requires equal lengths: {len(s1)} vs {len(s2)}")
    return sum(c1 != c2 for c1, c2 in zip(s1, s2))


def levenshtein(s1: str, s2: str) -> int:
    """Levenshtein distance."""
    return levenshtein_distance(s1, s2)


def jaro_winkler(s1: str, s2: str) -> float:
    """Jaro-Winkler similarity (1.0 = identical)."""
    return jaro_winkler_similarity(s1, s2)


def lcs_length(s1: str, s2: str) -> int:
    """Longest Common Subsequence length."""
    matcher = difflib.SequenceMatcher(None, s1, s2)
    return sum(block.size for block in matcher.get_matching_blocks())

In [23]:
def levenshtein_distance_bootleg(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance_bootleg(s2, s1)
    
    if len(s2) == 0:
        return len(s1)
    
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]



In [22]:
# character error rate
# word error rate

def cer(reference: str, hypothesis: str) -> float:
    """
    Character Error Rate = Levenshtein(ref, hyp) / len(ref)
    Lower is better (0.0 = perfect).
    """
    if len(reference) == 0:
        return 0.0 if len(hypothesis) == 0 else 1.0
    return levenshtein_distance(reference, hypothesis) / len(reference)


def wer(reference: str, hypothesis: str) -> float:
    """
    Word Error Rate = Levenshtein(ref_words, hyp_words) / len(ref_words)
    Lower is better (0.0 = perfect).
    """
    ref_words  = reference.split()
    hyp_words  = hypothesis.split()
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    return levenshtein_distance(ref_words, hyp_words) / len(ref_words)

In [42]:
def levenshtein_bootleg2(s1, s2):
    if len(s1) > len(s2):
        s1, s2 = s2, s1
    
    if len(s2) == 0:
        return len(s1)
    
    # Initialize matrix
    matrice = []
    for i in range(len(s1) + 1):
        row = [0] * (len(s2) + 1)
        matrice.append(row)
    
    # Fill the matrix
    for i in range(len(s1) + 1):
        for j in range(len(s2) + 1):
            if i == 0:
                matrice[i][j] = j  # Empty string to j chars = j insertions
            elif j == 0:
                matrice[i][j] = i  # i chars to empty string = i deletions
            else:
                deletion = matrice[i-1][j] + 1           # Remove from s1
                insertion = matrice[i][j-1] + 1          # Add to s1
                substitution = matrice[i-1][j-1] + (0 if s1[i-1] == s2[j-1] else 1)
                
                matrice[i][j] = min(deletion, insertion, substitution)
    
    return matrice[len(s1)][len(s2)]

print(levenshtein_bootleg2("kitten", "sitting"))  # Output: 3

3


0 0 0 
0 0 0 
